# 気象庁 過去の気象データ 一括ダウンロード

[観測所ファインダー](https://awg-yk.github.io/weather-station-finder/) で出力した
**地点リストCSV**を使って、気象庁の過去の気象データをまとめて取得するノートブックです。

## 使い方（3ステップ）
1. 観測所ファインダーで地点を絞り込み、「選択結果をCSVでダウンロード」でCSVを保存
2. 下のセルを実行（▶ボタン）し、案内に従って **そのCSVをアップロード**
3. データの種類・観測項目・期間を選ぶと、自動でダウンロードが進み、最後にZIPが手元に落ちてきます

> 気象庁サーバーに負荷をかけないよう、1件ずつスリープを挟んで取得します。間隔は3秒以上を推奨します。


In [ ]:
# ============================================================
# 気象庁 過去の気象データ 一括ダウンロード（Google Colab・初心者向け）
#
# 観測所ファインダー( https://awg-yk.github.io/weather-station-finder/ )で
# 出力したCSVを入力に、指定した種類・項目・期間のデータをまとめて取得します。
#
# 使い方:
#   1. このセル全体をColabの新しいセルに貼り付けて実行する
#   2. 画面の案内に沿って順番に選ぶだけ:
#        ① 地点リストCSVをアップロード
#        ② データの種類（時別値/日別値/月別値 …）を選ぶ
#        ③ 観測項目（気温・降水量 …）を選ぶ
#        ④ 期間（開始日・終了日）を入力
#        ⑤ リクエスト間隔を入力
#        → 件数・推定時間を確認してから開始
#   3. 自動でダウンロードが進み、最後にZIPファイルが手元に落ちてくる
#
# 選べる「データの種類」と「観測項目」は、気象庁の公式ページ
#   https://www.data.jma.go.jp/risk/obsdl/index.php
# の選択画面と同じ内容・同じコード体系にそろえてあります。
#
# 特徴:
#   - 地点番号(prec_no/block_no)はCSVの「気象庁ページURL」列から直接読むため、
#     追加のマスタデータ取得は不要（無い場合のみGitHubのマスタで補完）。
#   - 既に取得済みのファイルは自動スキップするので、途中で切れても再実行で続きから。
#   - 混雑時のエラーには自動リトライ。CSV以外が返った場合は失敗として記録。
#   - 気象庁サーバーに負荷をかけないよう、1リクエストずつスリープを挟んで取得します。
# ============================================================

!pip install -q requests

import csv as csv_module
import json
import re
import shutil
import time
from dataclasses import asdict, dataclass, field
from datetime import date, timedelta
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from urllib.parse import parse_qs, urlparse

import requests
from google.colab import files

ROOT_URL = "https://www.data.jma.go.jp/risk/obsdl/index.php"
SHOW_URL = "https://www.data.jma.go.jp/risk/obsdl/show/table"
STATIONS_JSON_URL = "https://raw.githubusercontent.com/awg-yk/test/main/data/stations.json"

MAX_RETRIES = 3          # 1リクエストあたりの最大試行回数
LARGE_JOB_WARN = 200     # このリクエスト数を超えたら注意喚起

# ------------------------------------------------------------
# 集計期間（データの種類）: (コード, 表示名, 1リクエストで取る分割単位)
#   "month": 1か月ずつ  "year": 1年ずつ
#   時別値は1日あたりのデータ量が多いので月単位で分割する
# ------------------------------------------------------------
PERIOD_OPTIONS: List[Tuple[str, str, str]] = [
    ("9", "時別値",     "month"),
    ("1", "日別値",     "year"),
    ("2", "半旬別値",   "year"),
    ("4", "旬別値",     "year"),
    ("5", "月別値",     "year"),
    ("6", "3か月別値",  "year"),
]

# ------------------------------------------------------------
# 観測項目マスタ（気象庁公式ページから取得したものと同一）
#   時別値(9)は独立した項目セット。それ以外は共通セットで、各項目が対応する
#   期間コードを "kikan" として持つ。"category" は観測所ファインダーの
#   「観測要素」列（気温/降水量/風/日照時間/積雪/湿度）との突き合わせ用。
# ------------------------------------------------------------
ELEMENTS_HOURLY: List[Tuple[str, str, str]] = [
    ("201", "気温", "気温"),
    ("101", "降水量（前1時間）", "降水量"),
    ("301", "風向・風速", "風"),
    ("401", "日照時間（前1時間）", "日照時間"),
    ("610", "全天日射量（前1時間）", "日照時間"),
    ("501", "積雪の深さ", "積雪"),
    ("503", "降雪の深さ（前1時間）", "積雪"),
    ("605", "相対湿度", "湿度"),
    ("604", "蒸気圧", "湿度"),
    ("612", "露点温度", "湿度"),
    ("601", "現地気圧", "湿度"),
    ("602", "海面気圧", "湿度"),
    ("607", "雲量", ""),
    ("703", "天気", ""),
    ("704", "視程", ""),
]

# (コード, 表示名, 対応期間コード集合, category)
ELEMENTS_OTHER: List[Tuple[str, str, set, str]] = [
    ("201", "平均気温",            {"1", "2", "4", "5", "6"}, "気温"),
    ("202", "最高気温",            {"1", "2", "4", "5", "6"}, "気温"),
    ("203", "最低気温",            {"1", "2", "4", "5", "6"}, "気温"),
    ("204", "日最高気温の平均",    {"2", "4", "5", "6"}, "気温"),
    ("206", "日最低気温の平均",    {"2", "4", "5", "6"}, "気温"),
    ("205", "日最高気温の最低",    {"2", "4", "5", "6"}, "気温"),
    ("207", "日最低気温の最高",    {"2", "4", "5", "6"}, "気温"),
    ("101", "降水量の合計",        {"1", "2", "4", "5", "6"}, "降水量"),
    ("102", "日降水量の最大",      {"2", "4", "5", "6"}, "降水量"),
    ("401", "日照時間",            {"1", "2", "4", "5", "6"}, "日照時間"),
    ("610", "合計全天日射量",      {"1", "2", "4", "5", "6"}, "日照時間"),
    ("501", "最深積雪",            {"1", "2", "4", "5", "6"}, "積雪"),
    ("503", "降雪量の合計",        {"1", "2", "4", "5", "6"}, "積雪"),
    ("504", "降雪量日合計の最大",  {"2", "4", "5", "6"}, "積雪"),
    ("301", "平均風速",            {"1", "2", "4", "5", "6"}, "風"),
    ("302", "最大風速（風向）",    {"1", "2", "4", "5", "6"}, "風"),
    ("304", "最大瞬間風速（風向）", {"1", "2", "4", "5", "6"}, "風"),
    ("305", "最多風向",            {"1", "2", "4", "5", "6"}, "風"),
    ("605", "平均相対湿度",        {"1", "2", "4", "5", "6"}, "湿度"),
    ("606", "最小相対湿度",        {"1", "2", "4", "5", "6"}, "湿度"),
    ("604", "平均蒸気圧",          {"1", "2", "4", "5", "6"}, "湿度"),
    ("601", "平均現地気圧",        {"1", "2", "4", "5", "6"}, "湿度"),
    ("602", "平均海面気圧",        {"1", "2", "4", "5", "6"}, "湿度"),
    ("603", "最低海面気圧",        {"1", "2", "4", "5", "6"}, "湿度"),
    ("607", "平均雲量",            {"1", "2", "4", "5", "6"}, ""),
    ("701", "天気概況（昼）",      {"1"}, ""),
    ("702", "天気概況（夜）",      {"1"}, ""),
]

# 入力CSVで「観測所ID」に相当し得る列名（サイトのバージョン差に対応）
ID_COLUMN_CANDIDATES = ["観測所ID", "地点コード", "地点番号"]


@dataclass
class Station:
    station_id: str
    name: str
    prec_no: str
    block_no: str
    station_type: str   # "アメダス" or "気象官署"
    elements: set       # 観測している要素カテゴリ集合
    status: str         # "現役" / "廃止" など

    def station_num(self) -> str:
        if self.station_type == "気象官署":
            return "s" + self.block_no
        return "a" + self.block_no.zfill(4)


@dataclass
class WeatherDataPayload:
    stationNumList: List[str] = field(default_factory=list)
    aggrgPeriod: int = 1
    elementNumList: List[List[str]] = field(default_factory=list)
    interAnnualType: int = 1
    ymdList: List[str] = field(default_factory=list)  # [y1, y2, m1, m2, d1, d2]
    optionNumList: List[Any] = field(default_factory=list)
    downloadFlag: str = "true"
    rmkFlag: int = 1
    disconnectFlag: int = 1
    youbiFlag: int = 0
    fukenFlag: int = 0
    kijiFlag: int = 0
    huukouFlag: int = 0
    csvFlag: int = 1
    jikantaiFlag: int = 0
    jikantaiList: List[Any] = field(default_factory=list)
    ymdLiteral: int = 1

    def to_post_data(self) -> dict:
        data = {}
        for key, value in asdict(self).items():
            data[key] = json.dumps(value) if isinstance(value, list) else value
        return data


# ------------------------------------------------------------
# 入力CSVの読み込み（地点番号はURL列から直接取得）
# ------------------------------------------------------------
def parse_prec_block_from_url(url: str) -> Tuple[Optional[str], Optional[str]]:
    if not url:
        return None, None
    try:
        q = parse_qs(urlparse(url).query)
        prec = q.get("prec_no", [None])[0]
        block = q.get("block_no", [None])[0]
        return prec, block
    except Exception:
        return None, None


def parse_elements_cell(cell: str) -> set:
    if not cell:
        return set()
    return {p.strip() for p in re.split(r"[\/／,、]", cell) if p.strip()}


def load_master_fallback() -> Dict[str, dict]:
    """CSVにURL列が無い古い形式のための保険。GitHubのマスタを取得。"""
    cache = Path("stations_master.json")
    if not cache.exists():
        print("（URL列が無いため観測所マスタで補完します…取得中）")
        resp = requests.get(STATIONS_JSON_URL, timeout=30)
        resp.raise_for_status()
        cache.write_bytes(resp.content)
    raw = json.loads(cache.read_text(encoding="utf-8"))
    return {str(s["id"]): s for s in raw["stations"]}


def find_id_column(fieldnames: List[str]) -> Optional[str]:
    for cand in ID_COLUMN_CANDIDATES:
        if cand in fieldnames:
            return cand
    return None


def read_stations(input_csv: Path) -> List[Station]:
    with input_csv.open(encoding="utf-8-sig", newline="") as f:
        reader = csv_module.DictReader(f)
        fields = reader.fieldnames or []
        id_col = find_id_column(fields)
        if id_col is None:
            raise SystemExit(
                f"CSVに地点IDの列（{ '／'.join(ID_COLUMN_CANDIDATES) }）が見つかりません。列: {fields}"
            )
        rows = list(reader)

    has_url = "気象庁ページURL" in fields
    master = None if has_url else load_master_fallback()

    out: List[Station] = []
    for row in rows:
        sid = (row.get(id_col) or "").strip()
        if not sid:
            continue
        name = (row.get("地点名") or "").strip()
        stype = (row.get("種別") or "").strip()
        status = (row.get("状態") or "").strip()
        elems = parse_elements_cell(row.get("観測要素", ""))

        prec = block = None
        if has_url:
            prec, block = parse_prec_block_from_url(row.get("気象庁ページURL", ""))
        if (not prec or not block) and master is not None:
            m = master.get(sid)
            if m:
                prec, block = m.get("precNo"), m.get("blockNo")
                if not stype:
                    stype = m.get("stationType", "")
        if not prec or not block:
            print(f"  [スキップ] {name or sid}: 地点番号(prec_no/block_no)を特定できません")
            continue
        if not stype:
            # 種別不明時はblock_no桁数で推定（気象官署は概ね5桁の47xxx）
            stype = "気象官署" if len(block) >= 5 and block.startswith("47") else "アメダス"

        out.append(Station(sid, name or sid, prec, block, stype, elems, status))
    return out


# ------------------------------------------------------------
# 期間の分割
# ------------------------------------------------------------
def date_chunks(start: date, end: date, unit: str):
    cur = start
    while cur <= end:
        if unit == "month":
            if cur.month == 12:
                last = date(cur.year, 12, 31)
            else:
                last = date(cur.year, cur.month + 1, 1) - timedelta(days=1)
            chunk_end = min(last, end)
            nxt_month = cur.month + 1
            nxt_year = cur.year + (1 if nxt_month > 12 else 0)
            nxt_month = 1 if nxt_month > 12 else nxt_month
            nxt = date(nxt_year, nxt_month, 1)
        else:
            chunk_end = min(date(cur.year, 12, 31), end)
            nxt = date(cur.year + 1, 1, 1)
        yield cur, chunk_end
        cur = nxt


# ------------------------------------------------------------
# 取得（リトライ＋エラーHTML検出つき）
# ------------------------------------------------------------
def looks_like_csv(content: bytes) -> bool:
    head = content[:200].lstrip()
    if head[:1] == b"<":              # HTMLエラーページ
        return False
    return len(content) > 0


def fetch_chunk(session, station_num, chunk_start, chunk_end, aggrg_period, elements, sleep_sec):
    payload = WeatherDataPayload(
        stationNumList=[station_num],
        aggrgPeriod=int(aggrg_period),
        elementNumList=[[code, ""] for code in elements],
        ymdList=[
            str(chunk_start.year), str(chunk_end.year),
            str(chunk_start.month), str(chunk_end.month),
            str(chunk_start.day), str(chunk_end.day),
        ],
    )
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.post(SHOW_URL, data=payload.to_post_data(),
                                headers={"Referer": ROOT_URL}, timeout=60)
            resp.raise_for_status()
            if not looks_like_csv(resp.content):
                raise RuntimeError("CSV以外の応答（混雑またはデータ量超過の可能性）")
            return resp.content
        except Exception as e:
            last_err = e
            if attempt < MAX_RETRIES:
                time.sleep(sleep_sec * attempt)  # 少しずつ間隔を広げて再試行
    raise last_err


def save_csv(content: bytes, output_path: Path) -> None:
    try:
        output_path.write_text(content.decode("cp932"), encoding="utf-8-sig")
    except UnicodeDecodeError:
        output_path.write_bytes(content)


# ------------------------------------------------------------
# 対話式の入力
# ------------------------------------------------------------
def ask_date(prompt: str, default: str) -> date:
    while True:
        raw = input(f"{prompt} [既定値: {default}]: ").strip() or default
        try:
            return date.fromisoformat(raw)
        except ValueError:
            print("  形式が正しくありません。YYYY-MM-DD の形で入力してください（例: 2023-01-01）")


def choose_period() -> Tuple[str, str, str]:
    print("② データの種類を選んでください（番号を入力）")
    for i, (code, label, _unit) in enumerate(PERIOD_OPTIONS, 1):
        print(f"   {i}. {label}")
    while True:
        raw = input("番号 [既定値: 2（日別値）]: ").strip() or "2"
        if raw.isdigit() and 1 <= int(raw) <= len(PERIOD_OPTIONS):
            return PERIOD_OPTIONS[int(raw) - 1]
        print("  一覧の番号を入力してください")


def choose_elements(period_code: str) -> List[Tuple[str, str, str]]:
    """戻り値: [(code, label, category), ...]"""
    if period_code == "9":
        available = [(v, lbl, cat) for v, lbl, cat in ELEMENTS_HOURLY]
    else:
        available = [(v, lbl, cat) for v, lbl, kikan, cat in ELEMENTS_OTHER if period_code in kikan]

    print()
    print("③ 観測項目を選んでください（番号をカンマ区切りで複数指定できます）")
    for i, (code, label, cat) in enumerate(available, 1):
        print(f"   {i:>2}. {label}")
    temp_idx = next((i for i, (c, l, cat) in enumerate(available, 1) if "気温" in l), 1)
    rain_idx = next((i for i, (c, l, cat) in enumerate(available, 1) if "降水" in l), None)
    default_nums = str(temp_idx) + ("," + str(rain_idx) if rain_idx else "")

    while True:
        raw = input(f"番号（例: 1,2） [既定値: {default_nums}]: ").strip() or default_nums
        try:
            picks = [int(x) for x in raw.replace("、", ",").split(",") if x.strip()]
            if picks and all(1 <= p <= len(available) for p in picks):
                return [available[p - 1] for p in picks]
        except ValueError:
            pass
        print("  一覧の番号をカンマ区切りで入力してください")


def confirm(prompt: str) -> bool:
    return (input(prompt).strip().lower() or "y") in ("y", "yes", "はい")


# ============================================================
# ここから対話式に進みます
# ============================================================
print("① 観測所ファインダーで出力した地点リストCSVをアップロードしてください")
uploaded = files.upload()
input_csv = Path(next(iter(uploaded.keys())))
print()

stations = read_stations(input_csv)
if not stations:
    raise SystemExit("有効な地点がCSVから読み取れませんでした。")

period_code, period_label, chunk_unit = choose_period()
elements = choose_elements(period_code)           # [(code,label,category)]
element_codes = [e[0] for e in elements]
selected_categories = {e[2] for e in elements if e[2]}

print()
start = ask_date("④ 取得したい期間の開始日を入力してください", "2023-01-01")
end = ask_date("   取得したい期間の終了日を入力してください", str(date.today()))
if start > end:
    raise SystemExit("開始日は終了日より前にしてください")

sleep_raw = input("⑤ リクエスト間隔(秒)。気象庁サーバーへの配慮のため3秒以上を推奨 [既定値: 3]: ").strip()
sleep_sec = float(sleep_raw) if sleep_raw else 3.0

# --- 事前チェックと確認 ---
chunks = list(date_chunks(start, end, chunk_unit))
n_requests = len(stations) * len(chunks)
est_min = n_requests * sleep_sec / 60.0

discontinued = [s for s in stations if s.status and s.status != "現役"]
# 観測項目ミスマッチ（選んだカテゴリを観測要素に持たない地点）
mismatch = []
if selected_categories:
    for s in stations:
        if s.elements and not (selected_categories & s.elements):
            mismatch.append(s.name)

print()
print("── 設定内容 ──")
print(f"  データの種類 : {period_label}")
print(f"  観測項目     : {', '.join(e[1] for e in elements)}")
print(f"  期間         : {start} 〜 {end}（{'1か月' if chunk_unit == 'month' else '1年'}ごとに分割）")
print(f"  対象地点数   : {len(stations)} 地点")
print(f"  リクエスト数 : 約 {n_requests} 回（間隔 {sleep_sec}秒 → 推定 約 {est_min:.1f} 分）")
if discontinued:
    print(f"  ⚠ 廃止済み地点が {len(discontinued)} 件含まれます（期間によっては空データになる場合あり）")
if mismatch:
    ex = "、".join(mismatch[:5]) + ("…" if len(mismatch) > 5 else "")
    print(f"  ⚠ 選んだ項目を観測していない可能性のある地点が {len(mismatch)} 件（例: {ex}）")
print()

if n_requests >= LARGE_JOB_WARN:
    print(f"※ リクエスト数が多め（{n_requests}回）です。地点や期間を絞ることも検討してください。")
if not confirm("この内容で開始しますか？ [Y/n]: "):
    raise SystemExit("中止しました。設定を変えて再実行してください。")

# --- ダウンロード実行 ---
output_dir = Path("jma_data")
output_dir.mkdir(exist_ok=True)

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"})
session.get(ROOT_URL, timeout=30)
time.sleep(sleep_sec)

n_ok = n_skip = n_fail = 0
failures: List[str] = []

print()
for s in stations:
    print(f"取得中: {s.name} ({s.station_num()})")
    for chunk_start, chunk_end in chunks:
        out_path = output_dir / f"{s.name}_{period_label}_{chunk_start.isoformat()}_{chunk_end.isoformat()}.csv"
        if out_path.exists() and out_path.stat().st_size > 0:
            print(f"  スキップ(取得済): {out_path.name}")
            n_skip += 1
            continue
        try:
            content = fetch_chunk(session, s.station_num(), chunk_start, chunk_end,
                                  period_code, element_codes, sleep_sec)
            save_csv(content, out_path)
            print(f"  保存: {out_path.name}")
            n_ok += 1
        except Exception as e:
            print(f"  [エラー] {s.name} {chunk_start}〜{chunk_end}: {e}")
            failures.append(f"{s.name} {chunk_start}〜{chunk_end}: {e}")
            n_fail += 1
        time.sleep(sleep_sec)

# --- サマリ ---
print()
print("── 結果サマリ ──")
print(f"  成功: {n_ok} / スキップ(取得済): {n_skip} / 失敗: {n_fail}")
if failures:
    print("  失敗した項目:")
    for f in failures:
        print(f"    - {f}")

if n_ok > 0 or n_skip > 0:
    print()
    print("ZIPにまとめてダウンロードします...")
    shutil.make_archive("jma_data_result", "zip", output_dir)
    files.download("jma_data_result.zip")
else:
    print("保存できたファイルがないため、ZIPは作成しませんでした。")
